# 02 - Tool Design
The four tools the agent composes: **generator** (LLM), **executor** (sandbox), **test runner** (pytest), **debugger** (error analysis).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

## 1. Code generator

In [2]:
from src.tools.code_generator import generate_code, generate_tests
task = 'Remove duplicates from a list preserving order'
code = generate_code(task)
tests = generate_tests(code, task)
print(code); print(); print(tests)

2026-10-15 20:27:56,122 INFO llm role=generate target=gemini:gemini-3.5-flash-lite


2026-10-15 20:27:58,856 INFO llm role=tests target=gemini:gemini-3.5-flash-lite


def solution(lst):
    seen = set()
    return [x for x in lst if not (x in seen or seen.add(x))]

def test_no_duplicates():
    assert solution([1, 2, 3]) == [1, 2, 3]

def test_with_duplicates():
    assert solution([1, 2, 2, 3, 1, 4]) == [1, 2, 3, 4]

def test_empty_list():
    assert solution([]) == []

def test_all_duplicates():
    assert solution([5, 5, 5, 5]) == [5]

def test_mixed_types():
    assert solution([1, "a", 1, "b", "a", 2.5]) == [1, "a", "b", 2.5]


## 2. Executor (sandbox)
Static screen (imports, `eval`/`open`...), then an isolated `python -I` subprocess with a timeout.

In [3]:
from src.tools.code_executor import execute_code, screen_code
print(screen_code('import os'))
print(screen_code('x = eval("1")'))
print(screen_code(code))

(False, 'Restricted import: os')
(False, 'Restricted call: eval()')
(True, '')


## 3. Test runner
Structured pass/fail details from pytest output.

In [4]:
from src.tools.test_runner import run_tests
r = run_tests(code, tests)
print({k: r[k] for k in ('success', 'passed', 'failed', 'failed_tests')})

{'success': True, 'passed': 5, 'failed': 0, 'failed_tests': []}


## 4. Debugger
Parses a failure and builds the repair context handed back to the generator.

In [5]:
from src.tools.test_runner import run_tests
from src.tools.debugger import analyze_error, generate_fix_prompt
bad = 'def solution(xs):\n    return xs[0] / 0\n'
r = run_tests(bad, 'def test_a():\n    assert solution([1]) == 1\n')
print(analyze_error(r['error'], bad)); print(); print(generate_fix_prompt(r['error'], bad))

{'line': 3, 'error_type': 'ZeroDivisionError', 'message': 'division by zero', 'category': 'runtime_error'}

[runtime_error] ZeroDivisionError near line 3:
F                                                                        [100%]
================================== FAILURES ===================================
___________________________________ test_a ____________________________________

    def test_a():
>       assert solution([1]) == 1
               ^^^^^^^^^^^^^

test_solution.py:7: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _

xs = [1]

    def solution(xs):
>       return xs[0] / 0
               ^^^^^^^^^
E       ZeroDivisionError: division by zero

test_solution.py:3: ZeroDivisionError
=========================== short test summary info ===========================
FAILED test_solution.py::test_a - ZeroDivisionError: division by zero
1 failed in 0.30s


## 5. Closing the loop
The debugger output feeds `generate_code(..., error_context=..., previous_code=...)`.

In [6]:
fixed = generate_code('Return the first element divided by 1', generate_fix_prompt(r['error'], bad), bad)
print(fixed)

2026-10-15 20:28:06,707 INFO llm role=fix target=gemini:gemini-3.5-flash-lite


def solution(xs):
    return xs[0] / 1
